<a href="https://colab.research.google.com/github/human530/MD.Piece/blob/main/autoresearch_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Autoresearch on Colab
基於 [karpathy/autoresearch](https://github.com/karpathy/autoresearch)

**使用方式：** Runtime → Change runtime type → 選 GPU (T4 或更高)

> T4 (Turing) 也可以跑！本 notebook 會自動偵測 GPU 並 patch Flash Attention fallback。

In [1]:
# 確認 GPU 並偵測架構
!nvidia-smi
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print(f"\n✅ GPU: {name} (compute capability: {cap[0]}.{cap[1]})")
    if cap[0] >= 8:
        print("   Ampere+ → Flash Attention 3 原生支援")
    else:
        print("   Pre-Ampere → 將自動套用 SDPA fallback patch")
else:
    print("❌ 未偵測到 GPU！請到 Runtime → Change runtime type → 選 GPU")

Sun Mar 22 23:42:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 安裝 uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = os.path.expanduser('~/.local/bin') + ':' + os.environ['PATH']

downloading uv 0.10.12 x86_64-unknown-linux-gnu
no checksums to verify
installing to /usr/local/bin
  uv
  uvx
everything's installed!


In [3]:
# Clone autoresearch
!git clone https://github.com/karpathy/autoresearch.git
%cd autoresearch

Cloning into 'autoresearch'...
remote: Enumerating objects: 188, done.
remote: Counting objects: 100% (2/2), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 188 (delta 0), reused 0 (delta 0), pack-reused 186 (from 2)
Receiving objects: 100% (188/188), 529.07 KiB | 13.23 MiB/s, done.
Resolving deltas: 100% (102/102), done.
/content/autoresearch


In [4]:
# 安裝依賴
!uv sync

# 下載 T4 patch 腳本並套用（Pre-Ampere GPU 自動 patch）
!curl -sL https://raw.githubusercontent.com/human530/MD.Piece/claude/autoresearch-feature-ynJbT/colab_t4_patch.py -o /tmp/t4_patch.py
!python /tmp/t4_patch.py

Using CPython 3.10.12 interpreter at: /usr/bin/python3.10
Creating virtual environment at: .venv
Resolved 74 packages in 0.94ms
Prepared 67 packages in 1m 19s
Installed 67 packages in 701ms
 + annotated-doc==0.0.4
 + anyio==4.12.1
 + certifi==2026.2.25
 + charset-normalizer==3.4.4
 + click==8.3.1
 + contourpy==1.3.2
 + cycler==0.12.1
 + exceptiongroup==1.3.1
 + filelock==3.24.3
 + fonttools==4.61.1
 + fsspec==2026.2.0
 + h11==0.16.0
 + hf-xet==1.3.1
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.5.0
 + idna==3.11
 + jinja2==3.1.6
 + kernels==0.12.1
 + kiwisolver==1.4.9
 + markdown-it-py==4.0.0
 + markupsafe==3.0.3
 + matplotlib==3.10.8
 + mdurl==0.1.2
 + mpmath==1.3.0
 + networkx==3.4.2
 + numpy==2.2.6
 + nvidia-cublas-cu12==12.8.4.1
 + nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-runtime-cu12==12.8.90
 + nvidia-cudnn-cu12==9.10.2.21
 + nvidia-cufft-cu12==11.3.3.83
 + nvidia-cufile-cu12==1.13.1.3
 + nvidia-curand-cu12==10.3.9.90
 + nvidia

In [ ]:
# 資料準備（下載 + tokenizer，約 2 分鐘）
!uv run prepare.py --num-shards 10

Cache directory: /root/.cache/autoresearch

Data: downloading 11 shards (0 already exist)...
  Downloaded shard_00006.parquet
  Downloaded shard_00004.parquet
  Downloaded shard_00005.parquet
  Downloaded shard_00002.parquet
  Downloaded shard_00007.parquet
  Downloaded shard_00001.parquet
  Downloaded shard_00003.parquet
  Downloaded shard_00000.parquet
  Downloaded shard_00008.parquet
  Downloaded shard_00009.parquet
  Downloaded shard_06542.parquet
Data: 11/11 shards ready at /root/.cache/autoresearch/data

Tokenizer: training BPE tokenizer...


In [ ]:
# 跑一次訓練實驗（約 5 分鐘）
!uv run train.py 2>&1 | tee /tmp/run.log

# 提取結果
import re
log = open("/tmp/run.log").read()
bpb_match = re.findall(r"val_bpb[:\s]+([\d.]+)", log)
loss_match = re.findall(r"train_loss[:\s]+([\d.]+)", log)
step_match = re.findall(r"step[:\s]+(\d+)", log)

val_bpb = float(bpb_match[-1]) if bpb_match else None
train_loss = float(loss_match[-1]) if loss_match else None
steps = int(step_match[-1]) if step_match else None

print(f"\n{'='*40}")
print(f"val_bpb:    {val_bpb}")
print(f"train_loss: {train_loss}")
print(f"steps:      {steps}")
print(f"{'='*40}")

## 回傳結果到 Supabase（直連，不需 ngrok）

填入你的 Supabase URL 和 Key，結果會自動寫入資料庫。
可在 Supabase Dashboard → Settings → API 找到這些值。

In [ ]:
# Supabase 直連設定
!pip install supabase -q

SUPABASE_URL = ""  # @param {type:"string"}
SUPABASE_KEY = ""  # @param {type:"string"}
EXPERIMENT_NAME = "baseline-t4"  # @param {type:"string"}
NOTES = "T4 baseline run"  # @param {type:"string"}

import os
os.environ["SUPABASE_URL"] = SUPABASE_URL
os.environ["SUPABASE_KEY"] = SUPABASE_KEY

# 寫入 baseline 結果
from supabase import create_client
from datetime import datetime

sb = create_client(SUPABASE_URL, SUPABASE_KEY)
row = {
    "name": EXPERIMENT_NAME,
    "val_bpb": val_bpb,
    "train_loss": train_loss,
    "steps": steps,
    "notes": NOTES,
    "kept": True,
    "submitted_at": datetime.utcnow().isoformat(),
}
res = sb.table("experiments").insert(row).execute()
if res.data:
    print(f"✅ 已寫入 Supabase: {res.data[0].get('id', '?')}")
else:
    print(f"⚠️ 寫入失敗: {res}")

## 🔄 自動化實驗循環（Supabase 直連）

下載 `autoresearch_runner.py` 並自動跑多輪實驗（假設 → 修改 → 訓練 → 評估 → keep/revert）。
每輪結果自動寫入 Supabase，不需要後端或 ngrok。

In [ ]:
# 下載自動化 runner 並執行完整實驗循環（Supabase 直連）
!pip install supabase requests -q

# 下載最新 runner
!curl -sL https://raw.githubusercontent.com/human530/MD.Piece/claude/check-autoresearch-ETSCW/autoresearch_runner.py -o autoresearch_runner.py

# 設定參數
ROUNDS = 5  # @param {type:"integer"}

# 初始化 git（runner 需要 git commit/revert）
!git init . 2>/dev/null; git add -A; git commit -m "initial" --allow-empty 2>/dev/null

# 開始自動實驗循環（Supabase 直連，不需 ngrok！）
!python autoresearch_runner.py --rounds {ROUNDS} --supabase-url {SUPABASE_URL} --supabase-key {SUPABASE_KEY}

In [ ]:
# 查看結果摘要
import pandas as pd
df = pd.read_csv("results.tsv", sep="\t")
print(df.to_string(index=False))
print(f"\n🏆 Best: {df.loc[df['val_bpb'].idxmin(), 'name']} — val_bpb: {df['val_bpb'].min()}")